# Populate all hex maze decoding tables

Bulk-populate the hex maze decode tables for every hex maze session with a full-epoch decode in `DecodingOutput`. 

For a step-by-step walkthrough of a single session (and how to fetch / plot the
results), see `Hex_Maze_Decode_Tables.ipynb`. 

To check what's populated and sanity-check
the key sources, see `Verify_Hex_Maze_Tables.ipynb`.

Pipeline: detect every full-epoch decode -> populate `HexMazeDecodedPosition` -> populate the
downstream hex tables. 

## 1. Find all entries in DecodingOutput for hex maze epochs

In [3]:
import re
import pandas as pd

import spyglass.common as sgc
from spyglass.decoding.decoding_merge import DecodingOutput
from spyglass_hexmaze.hex_maze_behavior import HexMazeBlock

# Every hex maze session, and the run interval -> epoch map for each (from TaskEpoch)
hex_sessions = set(HexMazeBlock.fetch('nwb_file_name'))
session_keys = [{'nwb_file_name': s} for s in sorted(hex_sessions)]

run_intervals = {}
for row in (sgc.TaskEpoch & session_keys).fetch(
    'nwb_file_name', 'epoch', 'interval_list_name', as_dict=True
):
    run_intervals.setdefault(row['nwb_file_name'], {})[row['interval_list_name']] = row['epoch']


def decoding_interval_epoch(nwb_file_name, decoding_interval):
    """Which epoch a FULL-epoch decode belongs to, or None if it isn't one.

    We only populate full-epoch decodes here, so we only match the run interval:
      - the run interval itself: "00_r1" (Berke), "07_r4" (Frank)
      - a derived run interval: "01_r1_noPreTrialTimes" (run interval, pre-trial times removed)

    Sub-epoch decodes (e.g. "epoch3_block2", "epoch7_nonLocal_ALL",
    "epoch3_barrierShiftInterval1") return None and are skipped.
    """
    name = str(decoding_interval)
    for interval, epoch in run_intervals.get(nwb_file_name, {}).items():
        if name.startswith(interval):
            return epoch
    return None


# Walk every part of the DecodingOutput merge table and collect (merge_id, nwb, interval).
# We ask each part only for the attributes it has (all decoding parts carry decoding_interval).
decode_rows = []
for part in DecodingOutput().parts(as_objects=True):
    if 'nwb_file_name' not in part.heading.names:
        continue
    decode_rows.append(pd.DataFrame(
        part.fetch('merge_id', 'nwb_file_name', 'decoding_interval', as_dict=True)
    ))
decode_rows = pd.concat(decode_rows, ignore_index=True)
decode_rows = decode_rows[decode_rows['nwb_file_name'].isin(hex_sessions)]

# Build one HexMazeDecodedPosition key per decode
jobs = []
for _, r in decode_rows.iterrows():
    epoch = decoding_interval_epoch(r['nwb_file_name'], r['decoding_interval'])
    if epoch is None:
        # Un-comment if we want to print out all decoding intervals
        # print(f"skip (not a full-epoch decode): {r['nwb_file_name']}  [{r['decoding_interval']}]")
        continue
    jobs.append({
        'decoding_merge_id': str(r['merge_id']),
        'nwb_file_name': r['nwb_file_name'],
        'epoch': epoch,
    })

n_sessions = len({j['nwb_file_name'] for j in jobs})
print(f"Found {len(jobs)} decode job(s) across {n_sessions} hex maze sessions")

Found 56 decode job(s) across 29 hex maze sessions


## 2. Populate `HexMazeDecodedPosition`

We run 4 jobs in parallel to speed things up!

I am not sure what the memory constraints of this looks like yet - I'll increase/decrease this number if needed.

In [ ]:
from spyglass_hexmaze.hex_maze_decoding import HexMazeDecodedPosition

# Which jobs are already in HexMazeDecodedPosition? (safe to re-run after a kernel restart)
# decoding_merge_id comes back as a UUID, so cast to str to match our job keys.
existing = {
    (str(row['decoding_merge_id']), row['nwb_file_name'], row['epoch'])
    for row in HexMazeDecodedPosition.fetch(
        'decoding_merge_id', 'nwb_file_name', 'epoch', as_dict=True
    )
}
todo = [j for j in jobs
        if (j['decoding_merge_id'], j['nwb_file_name'], j['epoch']) not in existing]

print(f"{len(jobs) - len(todo)} already populated, {len(todo)} to go:")
for j in todo:
    print(f"  {j['nwb_file_name']}  epoch {j['epoch']}  ({j['decoding_merge_id'][:8]}...)")

# Populate in PARALLEL. Each entry loads the full posterior (~1-2 h and a lot of memory),
# so pick N_PROCESSES to match your machine -- more processes means more memory used at once.
# reserve_jobs=True coordinates the workers through the schema's jobs table, so no two
# processes ever grab the same key. That also makes it safe to run this cell from several
# kernels at once, or to interrupt and re-run -- finished (and reserved) entries are skipped.
N_PROCESSES = 4

if todo:
    errors = HexMazeDecodedPosition.populate(
        todo,                     # restrict the key_source to just our decode keys
        processes=N_PROCESSES,    # parallel workers
        reserve_jobs=True,        # coordinate workers; skip in-progress / done keys
        order="random",           # spread work so workers don't collide on the same session
        display_progress=True,
        suppress_errors=True,     # one bad entry won't kill the whole run
    )
    if errors:
        print(f"\n{len(errors)} entr(y/ies) errored (also logged in the schema jobs table):")
        for e in errors:
            print("  ", e)
    else:
        print("\nAll done.")
else:
    print("Nothing to populate.")

40 already populated, 16 to go:
  IM-1871_20250805_.nwb  epoch 0  (f5c9b308...)
  IM-1871_20250806_.nwb  epoch 0  (57737fca...)
  IM-1947_20260406_.nwb  epoch 0  (0fe23cc3...)
  Luna20250218_.nwb  epoch 1  (1e293d2e...)
  Toby20250316_.nwb  epoch 1  (caaf2072...)
  Toby20250318_.nwb  epoch 1  (71ce3fbf...)
  Toby20250318_.nwb  epoch 3  (2e2777bb...)
  Toby20250318_.nwb  epoch 5  (a974c52f...)
  Toby20250318_.nwb  epoch 7  (2ae3ecd3...)
  Toby20250319_.nwb  epoch 1  (ffa2f408...)
  Toby20250319_.nwb  epoch 3  (6fd140e6...)
  Toby20250319_.nwb  epoch 5  (0d7b5707...)
  Toby20250319_.nwb  epoch 7  (19e76f6e...)
  Toby20250327_.nwb  epoch 3  (6546074c...)
  Toby20250327_.nwb  epoch 5  (d0672658...)
  Toby20250327_.nwb  epoch 7  (dc7f9662...)


Processes:   0%|          | 0/16 [00:00<?, ?it/s][2026-07-15 10:59:23,605][WARNING]: Skipped checksum for file with hash: bb492261-3040-ff02-3a2d-fa65810b4f46, and path: /stelmo/nwb/analysis/IM-1871_20250805/IM-1871_20250805_95c1f405-f9be-4d73-a475-964bc242d371.nc
[2026-07-15 10:59:23,623][WARNING]: Skipped checksum for file with hash: 97ebfd22-9ad5-541e-99f5-aa3b745e6247, and path: /stelmo/nwb/analysis/IM-1947_20260406/IM-1947_20260406_daf6d19c-bc2d-423b-bd77-e55c5de897e1.nc
[2026-07-15 10:59:23,623][WARNING]: Skipped checksum for file with hash: 7596116b-9198-560b-8b16-cc2dd7f39bd0, and path: /stelmo/nwb/analysis/Toby20250318/Toby20250318_276709e9-e865-4c0a-bad1-bf78de6f6478.nc
[2026-07-15 10:59:23,625][WARNING]: Skipped checksum for file with hash: 41377c61-9105-7d14-a0b6-a634d203e002, and path: /stelmo/nwb/analysis/Toby20250318/Toby20250318_32e1e7d9-a87b-4d85-b920-679795e11a64.nc
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/xarray/namedarray/core.py:496: User

## 3. Populate the downstream hex decode tables

In [ ]:
from spyglass_hexmaze.hex_maze_decoding import (
    HexMazeDecodedPositionHex,
    HexMazeDecodedPositionHexAnnotated,
    HexMazeDecodedHexPath,
)

# Downstream tables inherit nwb_file_name/epoch from HexMazeDecodedPosition (+ HexCentroids),
# so their key_source is well-defined -- a plain .populate() fills whatever is now ready.
# (Sessions missing HexCentroids simply won't be populatable, which is what we want.)
# These are much faster than HexMazeDecodedPosition, but parallelize them too for consistency.
for table in (HexMazeDecodedPositionHex, HexMazeDecodedPositionHexAnnotated, HexMazeDecodedHexPath):
    table.populate(
        jobs,
        processes=N_PROCESSES,
        reserve_jobs=True,
        order="random",
        display_progress=True,
        suppress_errors=True,
    )

print()
print("Decode tables populated. Current counts for these jobs:")
print(f"  HexMazeDecodedPosition:             {len(HexMazeDecodedPosition & jobs)}")
print(f"  HexMazeDecodedPositionHex:          {len(HexMazeDecodedPositionHex & jobs)}")
print(f"  HexMazeDecodedPositionHexAnnotated: {len(HexMazeDecodedPositionHexAnnotated & jobs)}")
print(f"  HexMazeDecodedHexPath:              {len(HexMazeDecodedHexPath & jobs)}")

## Clear stuck populate jobs

With `reserve_jobs=True`, each worker marks its key as `reserved` in the schema's jobs table
while it runs. If a worker is **hard-killed** (not a clean error), that reservation is never
released, so `populate` keeps skipping the key forever. This cell shows the current
reservations / errors for the decode tables and lets you clear the stuck ones so they can
be retried. **Only clear when no populate is actively running.**

In [ ]:
import datajoint as dj
from spyglass_hexmaze.hex_maze_decoding import (
    HexMazeDecodedPosition,
    HexMazeDecodedPositionHex,
    HexMazeDecodedPositionHexAnnotated,
    HexMazeDecodedHexPath,
)

# The jobs table lives on the decode schema and tracks reserved / errored populate jobs
jobs = dj.Schema("hex_maze_decoding").jobs

decode_table_names = [t.table_name for t in (
    HexMazeDecodedPosition,
    HexMazeDecodedPositionHex,
    HexMazeDecodedPositionHexAnnotated,
    HexMazeDecodedHexPath,
)]
decode_jobs = jobs & [{"table_name": tn} for tn in decode_table_names]

reserved = decode_jobs & 'status = "reserved"'
errored = decode_jobs & 'status = "error"'
print(f"{len(decode_jobs)} job record(s) for the decode tables: "
      f"{len(reserved)} reserved, {len(errored)} error")
display(decode_jobs)

# Set to True to actually delete. A 'reserved' row whose worker crashed is never released,
# so populate keeps skipping that key -- deleting it lets populate retry. 'error' rows are
# failed attempts; delete them to retry those too. ONLY do this when no populate is running!
CLEAR_STUCK_JOBS = False

if CLEAR_STUCK_JOBS:
    stuck = decode_jobs & 'status in ("reserved", "error")'
    n = len(stuck)
    stuck.delete_quick()   # jobs has no dependents, so a quick (no-prompt) delete is fine
    print(f"Cleared {n} stuck/errored job record(s). Re-run the populate cell to retry them.")
else:
    print("CLEAR_STUCK_JOBS is False -- nothing deleted (set it to True to clear).")